In [14]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path
import matplotlib.patches as patches
import os
from PIL import Image
import random

def is_symmetric(vertices, axis='horizontal'):
    """
    Check if the shape is symmetric about the given axis.
    axis: 'horizontal' or 'vertical'
    """
    # Remove the last vertex (which is the same as the first for closed path)
    points = vertices[:-1]

    # Calculate the center of the grid (1.5, 1.5) for a 4x4 grid
    center = 1.5

    # Create a set of the original points for faster lookup
    original_points_set = {tuple(point) for point in points}

    # Check each point for its reflection
    for point in points:
        x, y = point

        if axis == 'horizontal':
            # Reflect across horizontal axis (y = 1.5)
            reflected_point = (x, 2 * center - y)
        else:  # vertical
            # Reflect across vertical axis (x = 1.5)
            reflected_point = (2 * center - x, y)

        # If the reflected point is not in the original set, shape is not symmetric
        if reflected_point not in original_points_set:
            return False

    return True

def generate_random_shape():
    """
    Generate a random shape and check if it's symmetric.
    Returns vertices and symmetry information.
    """
    # Create a 4x4 grid (for the center 200x200 area)
    grid_points = [(x, y) for x in range(4) for y in range(4)]

    # Choose a random number of vertices (between 4 and 8)
    num_vertices = np.random.randint(4, 9)

    # Randomly select points from the grid
    selected_indices = np.random.choice(len(grid_points), size=num_vertices, replace=False)
    vertices = np.array([grid_points[i] for i in selected_indices])

    # Sort points to create a convex hull-like shape
    # Compute the centroid and sort points by angle
    centroid = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - centroid[1], vertices[:, 0] - centroid[0])
    sorted_indices = np.argsort(angles)
    vertices = vertices[sorted_indices]

    # Create a closed path by connecting the vertices
    vertices = np.vstack([vertices, vertices[0]])  # Close the path

    # Check symmetry
    horizontal_symmetric = is_symmetric(vertices, 'horizontal')
    vertical_symmetric = is_symmetric(vertices, 'vertical')

    return vertices, horizontal_symmetric, vertical_symmetric

def save_shape(vertices, save_path):
    """
    Save the shape defined by vertices to the specified path.
    The shape will be centered in a 300x300 image, but only occupy the center 200x200 area.
    """
    # Create a figure with fixed size of 300x300 pixels
    fig, ax = plt.subplots(figsize=(3, 3), dpi=100)  # 3 inches * 100 dpi = 300 pixels

    # Create a Path object
    codes = [Path.MOVETO] + [Path.LINETO] * (len(vertices) - 2) + [Path.CLOSEPOLY]
    path = Path(vertices, codes)

    # Add the patch to the axis
    patch = patches.PathPatch(path, facecolor='white', edgecolor='black', alpha=0.8, lw=2)
    ax.add_patch(patch)

    # Set the limits to show the entire 300x300 area, but the shape will only be in the center 200x200
    # For a 300x300 image with the shape in the center 200x200:
    # - The 4x4 grid (0-3 on each axis) should map to the center 200x200 pixels
    # - This means we need 50 pixels of margin on each side
    # - In the 0-3 grid coordinates, 50 pixels = 0.5 units (since 3 units = 200 pixels)
    ax.set_xlim(-0.75, 3.75)  # Add 0.75 units (50 pixels) of margin on each side
    ax.set_ylim(-0.75, 3.75)

    # Remove axes, grid, and ticks for a clean look
    ax.set_axis_off()

    # Set background color to white
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')

    plt.tight_layout()

    # Save the figure with exact 300x300 pixels
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=100, facecolor='white')
    plt.close(fig)  # Close the figure to free memory

def generate_derived_images(input_dir, output_dir):
    """
    Generate 5 derived images for each original image in the input directory.
    Some will be just rotated, others will be mirrored and then rotated.
    Creates a label file indicating which images are just rotated (yes) vs mirrored+rotated (no).
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Get list of all PNG files in the input directory
    image_files = [f for f in os.listdir(input_dir) if f.endswith('.png')]

    # Create a label file
    label_file_path = os.path.join(output_dir, "labels.txt")

    with open(label_file_path, 'w') as label_file:
        for image_name in image_files:
            base_name = os.path.splitext(image_name)[0]
            input_path = os.path.join(input_dir, image_name)

            # Open the original image
            original_img = Image.open(input_path)

            # Generate 5 derived images
            for i in range(1, 6):
                derived_name = f"{base_name}-{i}.png"
                output_path = os.path.join(output_dir, derived_name)

                # Randomly decide whether to mirror the image or just rotate it
                # For variety, we'll use a 50% chance of mirroring
                mirror = random.choice([True, False])

                # Random rotation angle between 0 and 360 degrees
                rotation_angle = random.uniform(0, 360)

                # Create a copy of the original image to modify
                derived_img = original_img.copy()

                # Apply mirroring if selected
                if mirror:
                    # Randomly choose horizontal or vertical mirroring
                    if random.choice([True, False]):
                        derived_img = derived_img.transpose(Image.FLIP_LEFT_RIGHT)  # Horizontal mirror
                    else:
                        derived_img = derived_img.transpose(Image.FLIP_TOP_BOTTOM)  # Vertical mirror

                    label = "no"  # Mirrored and rotated
                else:
                    label = "yes"  # Only rotated

                # Apply rotation
                derived_img = derived_img.rotate(rotation_angle, resample=Image.BICUBIC, expand=False)

                # Save the derived image
                derived_img.save(output_path)

                # Write the label to the file (just yes or no)
                label_file.write(f"{label}\n")

                print(f"Generated {derived_name} - {'Mirrored+Rotated' if mirror else 'Rotated'} by {rotation_angle:.1f}°")

    print(f"All derived images have been saved to '{output_dir}'")
    print(f"Labels have been saved to '{label_file_path}'")

def generate_original_shapes():
    # Create output directory if it doesn't exist
    output_dir = "images"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Ask user how many images to generate
    num_images = int(input("How many random shapes would you like to generate? "))

    print(f"Generating {num_images} non-symmetric random shapes...")

    # Generate and save the images
    for i in range(num_images):
        save_path = os.path.join(output_dir, f"{i}.png")

        # Generate shapes until we find a non-symmetric one
        max_attempts = 100  # Limit the number of attempts to avoid infinite loops
        attempts = 0

        while attempts < max_attempts:
            vertices, h_sym, v_sym = generate_random_shape()

            # If shape is not symmetric in either direction, use it
            if not h_sym and not v_sym:
                break

            attempts += 1

        if attempts == max_attempts:
            print(f"Warning: Could not generate a non-symmetric shape after {max_attempts} attempts for image {i}.")

        # Save the shape
        save_shape(vertices, save_path)

        print(f"Generated image {i+1}/{num_images}: {save_path} (non-symmetric)")

    print(f"All images have been saved to the '{output_dir}' folder.")
    return output_dir

def main():
    # Step 1: Generate original random shapes
    print("Step 1: Generating original random shapes...")
    input_dir = generate_original_shapes()

    # Step 2: Generate derived images
    print("\nStep 2: Generating derived images from the original random shapes...")
    output_dir = "images"
    generate_derived_images(input_dir, output_dir)

    print("\nProcess completed successfully!")

if __name__ == "__main__":
    main()


Step 1: Generating original random shapes...
Generating 5 non-symmetric random shapes...
Generated image 1/5: images\0.png (non-symmetric)
Generated image 2/5: images\1.png (non-symmetric)
Generated image 3/5: images\2.png (non-symmetric)
Generated image 4/5: images\3.png (non-symmetric)
Generated image 5/5: images\4.png (non-symmetric)
All images have been saved to the 'images' folder.

Step 2: Generating derived images from the original random shapes...
Generated 0-1.png - Mirrored+Rotated by 263.8°
Generated 0-2.png - Rotated by 328.0°
Generated 0-3.png - Mirrored+Rotated by 228.4°
Generated 0-4.png - Rotated by 140.3°
Generated 0-5.png - Mirrored+Rotated by 8.9°
Generated 1-1.png - Mirrored+Rotated by 158.3°
Generated 1-2.png - Rotated by 142.7°
Generated 1-3.png - Rotated by 287.2°
Generated 1-4.png - Mirrored+Rotated by 59.2°
Generated 1-5.png - Mirrored+Rotated by 110.4°
Generated 2-1.png - Rotated by 62.1°
Generated 2-2.png - Mirrored+Rotated by 302.9°
Generated 2-3.png - Mirro